# Solar EoL Transportation Cost Estimator

#### Install necessary packages

In [1]:
!pip install beautifulsoup4 pandas folium requests ipywidgets geopy

#### Load packages

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import math
import ipywidgets as widgets
from IPython.display import display
import requests
import folium

#### Extract position of all solar panel power stations

In [2]:
# Load the HTML content from your saved .txt file
with open('power_stations.txt', 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file.read(), 'html.parser')

# Find the table containing the data
table = soup.find('table', class_='power-stations-data-table')
data = []

# Loop through every row in the table body
for row in table.find('tbody').find_all('tr'):
    cols = row.find_all('td')
    
    # Ensure the row has the correct number of columns
    if len(cols) >= 8:
        name = cols[0].text.strip()
        size = cols[3].text.strip()
        
        # Extract the raw link exactly as it is written in your HTML
        link_tag = cols[7].find('a')
        link = link_tag['href'] if link_tag and 'href' in link_tag.attrs else None
        
        lat, lon = None, None
        
        # Use Regex to extract the coordinates directly from the link string
        if link:
            coords_match = re.search(r'@([-]?\d+\.\d+),([-]?\d+\.\d+)', link)
            
            if coords_match:
                lat = float(coords_match.group(1))
                lon = float(coords_match.group(2))
        
        # Append the extracted info to our list
        data.append({
            'Name': name,
            'Size (kW)': size,
            'Latitude': lat,
            'Longitude': lon
        })

# Convert to a Pandas DataFrame
list_ps = pd.DataFrame(data)
list_ps

,Name,Size (kW),Latitude,Longitude
0,100 Harris Street Pyrmont,199.9,-33.868350,151.193742
1,110 Somersby Falls Rd,199.1,-33.410577,151.279734
2,115 Frederick Street,292.0,-27.387098,153.075908
3,1182 Old Port Rd Power Station,199.7,-34.863683,138.508196
4,123 Sippy Downs,199.6,-26.713876,153.061123
...,...,...,...,...
2873,Zammit Ham 2,281.1,-33.798161,150.956296
2874,ZENEXUS,250.3,-27.587940,152.878980
2875,Zerella Virginia,997.0,-34.639700,138.588900
2876,ZEUSAPPOLLO SOLAR TC Do,141.6,-31.351400,115.566100


#### List of Recycling Sites

In [3]:
list_rs = pd.read_csv('recycling_sites.csv')
list_rs

,Name,Address,Latitude,Longitude
0,Lotus Recycling,"Unit 1/164-170 Barry Rd, Campbellfield VIC 3061",-37.666821,144.961087
1,ElecSome Kilmany,"14A Velore Rd, Kilmany VIC 3851",-38.102452,146.922293
2,ElecSome Keysborough,"67 Naxos Way, Keysborough VIC 3173",-38.022727,145.184209
3,"Sircel E-waste Recycling Facility, Villawood NSW","82 Marple Ave, Villawood NSW 2163",-33.884851,150.981646
4,"Sircel E-waste Recycling Facility, Parkes NSW","55A Brolgan Rd, Parkes NSW 2870",-33.137845,148.155682
5,PanelCycle,"19-23 Fariola St, Silverwater NSW 2128",-33.830921,151.050608
6,SolaCycle,"2/109 Victoria Rd, Drummoyne NSW 2047",-33.846417,151.157124
7,A1 Metal Recycle,"38 Carrington Rd, Guildford NSW 2161",-33.852583,150.976268
8,SOLAREC SA,"221 Hanson Rd, Athol Park SA 5012",-34.838808,138.541794


#### List of Suppliers/Buyers

In [4]:
list_ss = pd.read_excel('suppliers.xlsx')
list_ss

,Name,Location,Material,Latitude,Longitude
0,Pan Pacific Recycling,"10-12 Magnesium Drive, Crestmead, QLD 4132, Au...",Aluminium,-26.215925,152.363336
1,Yennora Copper Recycling,"31 The Promenade, Yennora NSW 2161",Aluminium,-33.864135,150.978121
2,Highett Metal,"283–295 Boundary Rd, Mordialloc VIC 3195",Aluminium,-37.989055,145.106946
3,United Metal Recycling,"16 Clements Ave, Bundoora VIC 3083",Aluminium,-37.702622,145.073205
4,Super Metal Recycling NSW Pty Ltd,"9 Dunheved Cct, St Marys NSW 2760",Aluminium,-33.532936,150.701044
5,Super Metal Recycling,"345 Frankston–Dandenong Rd, Dandenong South VI...",Aluminium,-37.857918,145.148068
6,All Metals Scrap,"24 Manton Rd, Oakleigh South VIC 3167",Aluminium,-37.914473,145.110406
7,Pan Pacific Recycling,"10-12 Magnesium Drive, Crestmead, QLD 4132, Au...",Copper,-26.215925,152.363336
8,Yennora Copper Recycling,"31 The Promenade, Yennora NSW 2161",Copper,-33.864135,150.978121
9,Safari Copper Recycling,"1/108 Newton Rd, Wetherill Park NSW 2164",Copper,-33.843511,150.892562


#### Add more Power Stations, Recycling Sites, or Suppliers

Run the following cell if you want to add more power stations, recycling sites, and/or suppliers.

In [5]:
# Ask user what they want to add
print("What do you want to add?")
print("1. Power station")
print("2. Recycling site")
print("3. Supplier")
print("4. I don't want to add anything")

choice = input("Enter your choice (1, 2, 3, or 4): ")

# 1. Power station
if choice == '1':
    name = input("Enter Power Station name: ")
    size = float(input("Enter size (kW): "))
    lat = float(input("Enter Latitude: "))
    lon = float(input("Enter Longitude: "))
    
    # Add to list_ps
    new_data = pd.DataFrame([{"Name": name, "Size (kW)": size, "Latitude": lat, "Longitude": lon}])
    list_ps = pd.concat([list_ps, new_data], ignore_index=True)
    print(f"\nSuccessfully added Power Station: {name}")

# 2. Recycling site
elif choice == '2':
    name = input("Enter Recycling Site name: ")
    address = input("Enter Address: ")
    lat = float(input("Enter Latitude: "))
    lon = float(input("Enter Longitude: "))
    
    # Add to list_rs
    new_data = pd.DataFrame([{"Name": name, "Address": address, "Latitude": lat, "Longitude": lon}])
    list_rs = pd.concat([list_rs, new_data], ignore_index=True)
    print(f"\nSuccessfully added Recycling Site: {name}")

# 3. Supplier
elif choice == '3':
    name = input("Enter Supplier name: ")
    address = input("Enter Address: ")
    
    # Simulating a dropdown box with a numbered menu
    print("\nSelect the Material they supply:")
    print("1. Aluminium")
    print("2. Copper")
    print("3. Glass")
    print("4. Plastic/EVA")
    print("5. Silicon")
    print("6. Silver")
    
    # Dictionary to map numbers to the actual material strings
    material_options = {
        "1": "Aluminium", 
        "2": "Copper", 
        "3": "Glass", 
        "4": "Plastic/EVA", 
        "5": "Silicon", 
        "6": "Silver"
    }
    
    mat_choice = input("Enter the number of the material (1-6): ")
    
    # Get the material based on choice, default to Unknown if they type something wrong
    material = material_options.get(mat_choice, "Unknown")
    
    lon = float(input("Enter Longitude: "))
    lat = float(input("Enter Latitude: "))
    
    # Add to list_ss
    new_data = pd.DataFrame([{"Name": name, "Address": address, "Material": material, "Longitude": lon, "Latitude": lat}])
    list_ss = pd.concat([list_ss, new_data], ignore_index=True)
    print(f"\nSuccessfully added Supplier: {name} (Material: {material})")

# 4. Skip
elif choice == '4':
    print("\nNo new data added.")
    
else:
    print("Invalid choice. Please run the code again and enter 1, 2, or 3.")

What do you want to add?
1. Power station
2. Recycling site
3. Supplier
4. I don't want to add anything


Enter your choice (1, 2, 3, or 4):  4



No new data added.


## User Input for Calculation

### Choose a Power Station

In [6]:
# Clean the data
list_ps_clean = list_ps.dropna(subset=['Name', 'Latitude', 'Longitude']).drop_duplicates(subset=['Name'])

# Create a dictionary to quickly map Name -> {Latitude, Longitude}
places_dict = list_ps_clean.set_index('Name')[['Latitude', 'Longitude']].to_dict('index')

# Extract all names as a list of strings for the dropdown
all_places = [str(name) for name in places_dict.keys()]

# Create variables to store the coordinates
ps_lat = None
ps_lon = None

# Create the Searchable Dropdown
place_selector = widgets.Combobox(
    placeholder='Start typing a place name...',
    options=all_places,
    description='Location:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='400px')
)

# Create an output area for visual feedback
output = widgets.Output()

# Define the function that runs when a user selects a place
def on_place_change(change):
    global ps_lat, ps_lon 
    output.clear_output()
    selected_ps = change['new']
    
    # Check if the selection is valid
    if selected_ps in places_dict:
        # Extract and save coordinates
        ps_lat = places_dict[selected_ps]['Latitude']
        ps_lon = places_dict[selected_ps]['Longitude']
        
        # Display the result to the user
        with output:
            print(f"Power Station Selected: {selected_ps}")
            print(f"Latitude:  {ps_lat}")
            print(f"Longitude: {ps_lon}")

# Attach the function to the combobox
place_selector.observe(on_place_change, names='value')

# Display the interactive widget
display(place_selector, output)

Combobox(value='', description='Location:', ensure_option=True, layout=Layout(width='400px'), options=('100 Ha…

Output()

### How many kW/kg/number of panels user want to recycle?

In [7]:
# Initialise variables globally so other cells can access them
total_kw = None
total_panels = None
total_kg = None
choice = None
value = None

def convert_solar_data():
    # Tell the function to use the global variables instead of creating local ones
    global choice, value, total_kw, total_panels, total_kg
    
    print("--- Solar Module Unit Converter ---")
    print("Select the unit to input your data:")
    print("1: Kilowatts (kW)")
    print("2: Kilograms (kg)")
    print("3: Number of Panels")
    
    choice = input("Enter 1, 2, or 3: ")
    
    if choice not in ['1', '2', '3']:
        print("Invalid selection. Please restart.")
        return

    try:
        value = float(input("Enter the value: "))
    except ValueError:
        print("Invalid number. Please enter a numerical value.")
        return

    # Base Assumptions (300W Module)
    kw_per_panel = 0.300
    kg_per_panel = 18.0
    
    # Conversion Logic
    if choice == '1': # User inputted kW
        total_kw = value
        total_panels = total_kw / kw_per_panel
        total_kg = total_panels * kg_per_panel
        
    elif choice == '2': # User inputted kg
        total_kg = value
        total_panels = total_kg / kg_per_panel
        total_kw = total_panels * kw_per_panel
        
    elif choice == '3': # User inputted Number of Panels
        total_panels = value
        total_kw = total_panels * kw_per_panel
        total_kg = total_panels * kg_per_panel

    # Output Results
    print("\n--- Converted Values ---")
    print(f"Capacity:      {total_kw:,.2f} kW")
    print(f"Total Mass:    {total_kg:,.2f} kg")
    print(f"Total Panels:  {total_panels:,.0f} panels")

if __name__ == "__main__":
    convert_solar_data()

--- Solar Module Unit Converter ---
Select the unit to input your data:
1: Kilowatts (kW)
2: Kilograms (kg)
3: Number of Panels


Enter 1, 2, or 3:  1
Enter the value:  1066



--- Converted Values ---
Capacity:      1,066.00 kW
Total Mass:    63,960.00 kg
Total Panels:  3,553 panels


## Conventional Recycling

### Number of Truck Needed

In [8]:
# Truck specifications
truck_volume_limit = 28 # cubic meters
truck_weight_limit = 7000 # kg
truck_w, truck_l, truck_h = 2.2, 6.2, 2.4 # meters

# Module specifications
module_capacity_w = 300 # Watts
module_w, module_l, module_h = 1.0, 1.7, 0.035 # meters
module_weight = (module_capacity_w / 1000) * 60 # 18.0 kg
module_volume = module_w * module_l * module_h # 0.0595 cubic meters

print("Capacity of the truck:")
# Number of panels limit by volume
panels_by_volume = int(truck_volume_limit // module_volume)
print(f"Limit by pure volume: {panels_by_volume} panels")

# Number of panels limit by weight
panels_by_weight = int(truck_weight_limit // module_weight)
print(f"Limit by pure weight: {panels_by_weight} panels")

# Number of panels limit by physical space
stacks_wide = int(truck_w // module_w)
stacks_long = int(truck_l // module_l)
panels_high = int(truck_h // module_h)

panels_by_space = stacks_wide * stacks_long * panels_high
print(f"Limit by physical space: {panels_by_space} panels")

# Store the limits in a dictionary to link the names to the calculated values
limits = {
    "pure volume": panels_by_volume,
    "pure weight": panels_by_weight,
    "physical space": panels_by_space
}

limiting_factor = min(limits, key=limits.get)
max_panels = limits[limiting_factor]
print(f"\nConclusion: \nThe truck is limited by {limiting_factor}, it can carry a maximum of {max_panels} panels.")

# Determine the unit string based on the user's choice from the previous cell
if choice == '1':
    unit = "kW"
elif choice == '2':
    unit = "kg"
elif choice == '3':
    unit = "panels"

# Calculate the number of trucks needed (rounding up)
trucks_needed = math.ceil(total_panels / max_panels)

# Print the final output
print(f"If user want to recycle {value:g} {unit} of solar EoL, they need {trucks_needed} trucks.")

Capacity of the truck:
Limit by pure volume: 470 panels
Limit by pure weight: 388 panels
Limit by physical space: 408 panels

Conclusion: 
The truck is limited by pure weight, it can carry a maximum of 388 panels.
If user want to recycle 1066 kW of solar EoL, they need 10 trucks.


### Estimate Driving Distance and Optimal Route to Nearest Recycling Site

In [15]:
# Define the function to get driving distance and route geometry from OSRM
def get_osrm_driving_data(lat1, lon1, lat2, lon2):
    # OSRM API expects coordinates in longitude,latitude order
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
    
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['code'] == 'Ok':
            # Extract distance (OSRM returns meters, we convert to km)
            distance_km = data['routes'][0]['distance'] / 1000
            # Extract the shape of the route for mapping
            route_geometry = data['routes'][0]['geometry'] 
            return distance_km, route_geometry
            
    # If the API fails or no route is found, return infinity
    return float('inf'), None

# Initialise tracking variables
closest_site = None
min_driving_distance = float('inf')
best_route_geometry = None

# Loop through the sites and query OSRM for each
for index, site in list_rs.iterrows():
    site_lat = site['Latitude']
    site_lon = site['Longitude']
    
    # Calculate driving distance
    dist_km, geometry = get_osrm_driving_data(ps_lat, ps_lon, site_lat, site_lon)
    
    # Update if this driving route is shorter than our current best
    if dist_km < min_driving_distance:
        min_driving_distance = dist_km
        closest_site = site
        best_route_geometry = geometry

rs_lat = closest_site['Latitude']
rs_lon = closest_site['Longitude']

# Output the results
print(f"Closest Recycling Hub: {closest_site['Name']}")
print(f"Address: {closest_site['Address']}")
print(f"Driving Distance: {min_driving_distance:.2f} km")

# Create the Map
mid_lat = (ps_lat + closest_site['Latitude']) / 2
mid_lon = (ps_lon + closest_site['Longitude']) / 2
route_map = folium.Map(location=[mid_lat, mid_lon], zoom_start=12)

# Add Power Station Marker
folium.Marker(
    location=[ps_lat, ps_lon],
    popup="Selected Power Station",
    icon=folium.Icon(color="blue", icon="bolt", prefix='fa')
).add_to(route_map)

# Add Recycling Hub Marker
folium.Marker(
    location=[closest_site['Latitude'], closest_site['Longitude']],
    popup=f"Recycling Hub: {closest_site['Name']}",
    icon=folium.Icon(color="green", icon="recycle", prefix='fa')
).add_to(route_map)

# Draw the actual road route using the GeoJSON from OSRM
if best_route_geometry:
    folium.GeoJson(
        best_route_geometry,
        name="Driving Route",
        style_function=lambda feature: {
            'color': '#FF0000',
            'weight': 4,
            'opacity': 0.8
        },
        tooltip=f"Driving distance: {min_driving_distance:.2f} km"
    ).add_to(route_map)

# Display the map
route_map

Closest Recycling Hub: SolaCycle
Address: 2/109 Victoria Rd, Drummoyne NSW 2047
Driving Distance: 6.17 km


### Estimate Distance and Route to CLosest Suppliers for 6 Materials

In [10]:
# Initialise a dictionary to store the best supplier and route for each material
best_suppliers = {}

# Get the unique list of materials from the dataset (assuming there are 6)
materials = list_ss['Material'].unique()

# Define some distinct colors for the 6 different routes on the map
route_colors = ['#FF0000', '#0000FF', '#008000', '#800080', '#FFA500', '#00FFFF']

print("Finding the closest suppliers for each material...\n")

# Loop through each unique material
for i, material in enumerate(materials):
    # Filter the DataFrame for suppliers of this specific material
    material_suppliers = list_ss[list_ss['Material'] == material]
    
    min_dist = float('inf')
    best_supp = None
    best_geom = None
    
    # Loop through suppliers of this material to find the closest one
    for index, supp in material_suppliers.iterrows():
        supp_lat = supp['Latitude']
        supp_lon = supp['Longitude']
        
        # Call the existing OSRM function (using rs_lat and rs_lon from previous cell)
        dist, geom = get_osrm_driving_data(rs_lat, rs_lon, supp_lat, supp_lon)
        
        # Update if it's the closest one found so far for this material
        if dist < min_dist:
            min_dist = dist
            best_supp = supp
            best_geom = geom
            
    # Save the best result for this material
    if best_supp is not None:
        best_suppliers[material] = {
            'supplier': best_supp,
            'distance': min_dist,
            'geometry': best_geom,
            'color': route_colors[i % len(route_colors)]
        }

# Output the Results
print("CLOSEST SUPPLIERS IDENTIFIED:\n" + "-"*40)
for mat, data in best_suppliers.items():
    supp = data['supplier']
    print(f"Material: {mat}")
    print(f"Company: {supp['Name']}")
    print(f"Address: {supp['Location']}")
    print(f"Driving Distance: {data['distance']:.2f} km\n")

# Create the Map
# Center the map on the Recycling Site
suppliers_map = folium.Map(location=[rs_lat, rs_lon], zoom_start=9)

# Add Recycling Site Marker (Origin)
folium.Marker(
    location=[rs_lat, rs_lon],
    popup="Recycling Site (Origin)",
    icon=folium.Icon(color="green", icon="recycle", prefix='fa')
).add_to(suppliers_map)

# Loop through the best suppliers to add markers and routes to the map
for mat, data in best_suppliers.items():
    supp = data['supplier']
    geom = data['geometry']
    color = data['color']
    
    # Add Supplier Marker
    folium.Marker(
        location=[supp['Latitude'], supp['Longitude']],
        popup=f"<b>{supp['Name']}</b><br>Material: {mat}",
        icon=folium.Icon(color="orange", icon="industry", prefix='fa')
    ).add_to(suppliers_map)
    
    # Add Driving Route
    if geom:
        folium.GeoJson(
            geom,
            name=f"{mat} Route",
            # We use color=color in the lambda to ensure Python binds the current color in the loop correctly
            style_function=lambda feature, c=color: {
                'color': c,
                'weight': 4,
                'opacity': 0.8
            },
            tooltip=f"{mat} Route: {data['distance']:.2f} km"
        ).add_to(suppliers_map)

# Add layer control to toggle routes on/off
folium.LayerControl().add_to(suppliers_map)

# Display the map
suppliers_map

Finding the closest suppliers for each material...

CLOSEST SUPPLIERS IDENTIFIED:
----------------------------------------
Material: Aluminium
Company: Yennora Copper Recycling
Address: 31 The Promenade, Yennora NSW 2161
Driving Distance: 24.37 km

Material: Copper
Company: Austick Copper Recycling
Address: 12 Bellona Ave, Regents Park NSW 2143
Driving Distance: 19.40 km

Material: Glass
Company: Australian Stained Glass Supplies
Address: Unit 9/13/15 Wollongong Rd, Arncliffe NSW 2205
Driving Distance: 14.56 km

Material: Plastic / EVA
Company: Australian Plastic Fabricators
Address: Entry Via, Unit 2-15/42 Wattle St, Ultimo NSW 2007
Driving Distance: 7.11 km

Material: Silicon
Company: Trade Supply Direct
Address: Unit 16/40 Anzac St, Chullora NSW 2190
Driving Distance: 18.00 km

Material: Silver
Company: Ore Metal
Address: 115 Booth St, Annandale NSW 2038
Driving Distance: 5.72 km



### Estimate Transportation Cost

In [11]:
# Full Route: Recycling Site -> Power Station -> Recycling Site -> Suppliers -> Recycling Site
# Our assumptions
diesel_price_per_litre = 2.25
fuel_efficiency_L_per_100km = 17
fuel_efficiency_L_per_km = fuel_efficiency_L_per_100km / 100

# Path 1: Recycling Site -> Power Station -> Recycling Site
# Fuel Cost
round_trip_distance_km = min_driving_distance * 2
fuel_used_per_truck = round_trip_distance_km * fuel_efficiency_L_per_km
cost_per_truck = fuel_used_per_truck * diesel_price_per_litre
total_fuel_cost_1 = cost_per_truck * trucks_needed

# Labour Cost
driver_per_truck = 1
helper_per_truck = 2
driver_hourly_rate = 42
helper_hourly_rate = 45    
average_speed_kmh = 45     # Assumed constant speed in km/hr
onsite_work_hours = 4      # Time spent loading/unloading

driving_time_hours = round_trip_distance_km / average_speed_kmh
total_labour_cost_1 = ((driver_per_truck * driver_hourly_rate) + (helper_per_truck * helper_hourly_rate)) * trucks_needed * (driving_time_hours + onsite_work_hours)

# Total Cost
total_cost_1 = total_fuel_cost_1 + total_labour_cost_1

print("PATH 1: Recycling Site -> Power Station -> Recycling Site")
print("-" * 55)
print(f"Round-trip distance per truck: {round_trip_distance_km:.2f} km")
print(f"TOTAL FLEET FUEL COST: ${total_fuel_cost_1:.2f} AUD")
print(f"TOTAL FLEET LABOUR COST: ${total_labour_cost_1:.2f} AUD")
print(f"TOTAL PATH 1 COST: ${total_cost_1:.2f} AUD\n")

# Path 2: Recycling Site -> Suppliers -> Recycling Site
# Material composition assumptions
material_composition = {
    'Glass': 0.72,
    'Aluminium': 0.15,
    'Plastic / EVA': 0.07,
    'Silicon': 0.025,
    'Copper': 0.008,
    'Silver': 0.0003
}

total_fuel_cost_2 = 0
total_labour_cost_2 = 0

print("PATH 2: Recycling Site -> Suppliers -> Recycling Site")
print("-" * 55)

# Loop through the dictionary created in your Folium map code
for mat, data in best_suppliers.items():
    if mat not in material_composition:
        print(f"Warning: {mat} not in composition dictionary. Skipping.")
        continue
        
    # Calculate weight and trucks needed
    mat_kg = total_kg * material_composition[mat]
    trucks_for_mat = math.ceil(mat_kg / 7000)  # 7000 kg capacity per truck
    
    if trucks_for_mat == 0:
        continue # Skip if the weight is 0
        
    # Get route distance (from OSRM output stored in best_suppliers)
    mat_one_way_dist = data['distance']
    mat_round_trip_dist = mat_one_way_dist * 2
    
    # Fuel Cost for this material
    mat_fuel_used = mat_round_trip_dist * fuel_efficiency_L_per_km
    mat_fuel_cost = mat_fuel_used * diesel_price_per_litre * trucks_for_mat
    total_fuel_cost_2 += mat_fuel_cost
    
    # Labour Cost for this material
    mat_driving_hours = mat_round_trip_dist / average_speed_kmh
    mat_onsite_hours = 2 # 2 hours loading/unloading
    
    # Cost = (driver cost + helpers cost) * total hours per truck * number of trucks
    combined_hourly_rate = (driver_per_truck * driver_hourly_rate) + (helper_per_truck * helper_hourly_rate)
    mat_labour_cost = combined_hourly_rate * (mat_driving_hours + mat_onsite_hours) * trucks_for_mat
    total_labour_cost_2 += mat_labour_cost
    
    print(f"Material: {mat} ({mat_kg:.2f} kg)")
    print(f"  Trucks required: {trucks_for_mat}")
    print(f"  Round-trip distance: {mat_round_trip_dist:.2f} km")
    print(f"  Fuel: ${mat_fuel_cost:.2f} | Labour: ${mat_labour_cost:.2f}\n")

total_cost_2 = total_fuel_cost_2 + total_labour_cost_2

print(f"TOTAL PATH 2 FUEL COST: ${total_fuel_cost_2:.2f} AUD")
print(f"TOTAL PATH 2 LABOUR COST: ${total_labour_cost_2:.2f} AUD")
print(f"TOTAL PATH 2 COST: ${total_cost_2:.2f} AUD\n")

# Full Route Transportation Cost
total_project_cost = total_cost_1 + total_cost_2
print("=======================================================")
print(f"FULL ROUTE TOTAL COST (PATH 1 + PATH 2): ${total_project_cost:.2f} AUD")
print("=======================================================")

PATH 1: Recycling Site -> Power Station -> Recycling Site
-------------------------------------------------------
Round-trip distance per truck: 12.34 km
TOTAL FLEET FUEL COST: $47.20 AUD
TOTAL FLEET LABOUR COST: $5641.98 AUD
TOTAL PATH 1 COST: $5689.18 AUD

PATH 2: Recycling Site -> Suppliers -> Recycling Site
-------------------------------------------------------
Material: Aluminium (9594.00 kg)
  Trucks required: 2
  Round-trip distance: 48.75 km
  Fuel: $37.29 | Labour: $813.98

Material: Copper (511.68 kg)
  Trucks required: 1
  Round-trip distance: 38.79 km
  Fuel: $14.84 | Labour: $377.80

Material: Glass (46051.20 kg)
  Trucks required: 7
  Round-trip distance: 29.11 km
  Fuel: $77.95 | Labour: $2445.80

Material: Plastic / EVA (4477.20 kg)
  Trucks required: 1
  Round-trip distance: 14.22 km
  Fuel: $5.44 | Labour: $305.72

Material: Silicon (1599.00 kg)
  Trucks required: 1
  Round-trip distance: 36.01 km
  Fuel: $13.77 | Labour: $369.62

Material: Silver (19.19 kg)
  Trucks

### Estimate Worth of EoL Waste (in $) and Profit

In [12]:
# Based on our assumptions
panel_waste_worth = total_kg * 0.2 # AUD
print(f"Estimate total worth of recycled panel is ${panel_waste_worth:.2f}")

# Estimate Profit
profit = panel_waste_worth - total_project_cost
print(f"Estimate profit from conventional recycling is ${profit:.2f}")

Estimate total worth of recycled panel is $12792.00
Estimate profit from conventional recycling is $2338.68


## Mobile Recycling

### Find Closest Suppliers

In [13]:
# Initialise a dictionary to store the best supplier and route for each material
best_suppliers = {}

# Get the unique list of materials from the dataset (assuming there are 6)
materials = list_ss['Material'].unique()

# Define some distinct colors for the 6 different routes on the map
route_colors = ['#FF0000', '#0000FF', '#008000', '#800080', '#FFA500', '#00FFFF']

print("Finding the closest suppliers for each material...\n")

# Loop through each unique material
for i, material in enumerate(materials):
    # Filter the DataFrame for suppliers of this specific material
    material_suppliers = list_ss[list_ss['Material'] == material]
    
    min_dist = float('inf')
    best_supp = None
    best_geom = None
    
    # Loop through suppliers of this material to find the closest one
    for index, supp in material_suppliers.iterrows():
        supp_lat = supp['Latitude']
        supp_lon = supp['Longitude']
        
        # Call the existing OSRM function using the Power Station coordinates (ps_lat and ps_lon)
        dist, geom = get_osrm_driving_data(ps_lat, ps_lon, supp_lat, supp_lon)
        
        # Update if it's the closest one found so far for this material
        if dist < min_dist:
            min_dist = dist
            best_supp = supp
            best_geom = geom
            
    # Save the best result for this material
    if best_supp is not None:
        best_suppliers[material] = {
            'supplier': best_supp,
            'distance': min_dist,
            'geometry': best_geom,
            'color': route_colors[i % len(route_colors)]
        }

# Output the Results
print("CLOSEST SUPPLIERS IDENTIFIED:\n" + "-"*40)
for mat, data in best_suppliers.items():
    supp = data['supplier']
    print(f"Material: {mat}")
    print(f"Company: {supp['Name']}")
    print(f"Address: {supp['Location']}")
    print(f"Driving Distance: {data['distance']:.2f} km\n")

# Create the Map
# Center the map on the Power Station
suppliers_map = folium.Map(location=[ps_lat, ps_lon], zoom_start=9)

# Add Power Station Marker (Origin)
folium.Marker(
    location=[ps_lat, ps_lon],
    popup="Power Station (Origin)",
    icon=folium.Icon(color="red", icon="bolt", prefix='fa')
).add_to(suppliers_map)

# Loop through the best suppliers to add markers and routes to the map
for mat, data in best_suppliers.items():
    supp = data['supplier']
    geom = data['geometry']
    color = data['color']
    
    # Add Supplier Marker
    folium.Marker(
        location=[supp['Latitude'], supp['Longitude']],
        popup=f"<b>{supp['Name']}</b><br>Material: {mat}",
        icon=folium.Icon(color="orange", icon="industry", prefix='fa')
    ).add_to(suppliers_map)
    
    # Add Driving Route
    if geom:
        folium.GeoJson(
            geom,
            name=f"{mat} Route",
            # We use color=color in the lambda to ensure Python binds the current color in the loop correctly
            style_function=lambda feature, c=color: {
                'color': c,
                'weight': 4,
                'opacity': 0.8
            },
            tooltip=f"{mat} Route: {data['distance']:.2f} km"
        ).add_to(suppliers_map)

# Add layer control to toggle routes on/off
folium.LayerControl().add_to(suppliers_map)

# Display the map
suppliers_map

Finding the closest suppliers for each material...

CLOSEST SUPPLIERS IDENTIFIED:
----------------------------------------
Material: Aluminium
Company: Yennora Copper Recycling
Address: 31 The Promenade, Yennora NSW 2161
Driving Distance: 26.54 km

Material: Copper
Company: Austick Copper Recycling
Address: 12 Bellona Ave, Regents Park NSW 2143
Driving Distance: 21.56 km

Material: Glass
Company: Australian Stained Glass Supplies
Address: Unit 9/13/15 Wollongong Rd, Arncliffe NSW 2205
Driving Distance: 13.96 km

Material: Plastic / EVA
Company: Australian Plastic Fabricators
Address: Entry Via, Unit 2-15/42 Wattle St, Ultimo NSW 2007
Driving Distance: 1.72 km

Material: Silicon
Company: Jehbco Silicones
Address: 24 William St, Brookvale NSW 2100
Driving Distance: 17.39 km

Material: Silver
Company: Australian Jewellers Supplies - Sydney
Address: Dymocks Building, Suite 15, Level 1/428 George St, Sydney NSW 2000
Driving Distance: 2.29 km



### Estimate Labour and Transport Cost of Mobile System (1 Vehicle)